In [ ]:
# ============================================================
# Synthetic Step5 setup
#
# This notebook does NOT upload or read config.py.
# All synthetic Step5 settings are defined explicitly here
# to avoid confusion with the main ExpB SettingA1 pipeline.
# ============================================================

import os
import sys
import shutil
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_4"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Explicit synthetic Step5 config
# ------------------------------------------------------------

SYNTHETIC_EXPERIMENT_NAME = "pilot4_synthetic_step4_lr_5000"

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MODEL_TAG = "Qwen2_5_0_5B"

MAX_LENGTH = 768
RUN_MODE = "synthetic"

SOURCE_TYPE = "pilot4_synthetic_step4_lr_5000_step4T_style_implicit_transitive_probe_samples"
INPUT_TEXT_MODE = "paragraph_probe_samples"
FEATURE_ANCHOR = "probe_pair_subject_object"

STEP5_INPUT_DIR = os.path.join(
    DATA_DIR,
    "pilot4_synthetic_step4_lr_5000_step4T_input"
)

STEP5_OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "pilot4_synthetic_step4_lr_5000_step5_hidden_states_qwen2_5_0_5b"
)

STEP5_INPUT_JSON_PATTERN = "step4T_*_implicit_transitive_probe_samples.json"
STEP5_INPUT_JSONL_PATTERN = "step4T_*_implicit_transitive_probe_samples.jsonl"

PRINT_FILE_SUMMARY = True
PRINT_GLOBAL_SUMMARY = True
PRINT_EXAMPLES = False

# ------------------------------------------------------------
# 3. Use synthetic Step4T zip from Google Drive
# ------------------------------------------------------------

step4_zip_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/setting_synthetic/data_synthetic/pilot4_synthetic_step4_lr_5000.zip"
)

print("\nUsing synthetic Step4T zip from Google Drive:")
print(step4_zip_path)

if not step4_zip_path.exists():
    raise FileNotFoundError(f"Synthetic Step4T zip not found: {step4_zip_path}")

# ------------------------------------------------------------
# 4. Unzip synthetic Step4T output
# ------------------------------------------------------------

tmp_extract_dir = "/content/_synthetic_step4T_extract_tmp"

if os.path.exists(tmp_extract_dir):
    shutil.rmtree(tmp_extract_dir)

os.makedirs(tmp_extract_dir, exist_ok=True)

with zipfile.ZipFile(str(step4_zip_path), "r") as zf:
    zf.extractall(tmp_extract_dir)

# ------------------------------------------------------------
# 5. Copy Step5-compatible synthetic probe sample files
# ------------------------------------------------------------

jsonl_files = sorted(
    Path(tmp_extract_dir).rglob(STEP5_INPUT_JSONL_PATTERN)
)

json_files = sorted(
    Path(tmp_extract_dir).rglob(STEP5_INPUT_JSON_PATTERN)
)

if not jsonl_files and not json_files:
    print("\nAll files found in synthetic Step4T zip:")
    for p in sorted(Path(tmp_extract_dir).rglob("*")):
        if p.is_file():
            print("-", p.relative_to(tmp_extract_dir))

    raise FileNotFoundError(
        "No Step5-compatible synthetic files found. Expected files matching:\n"
        f"  {STEP5_INPUT_JSONL_PATTERN}\n"
        f"  {STEP5_INPUT_JSON_PATTERN}\n"
        "Recommended filename:\n"
        "  step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json"
    )

input_files_to_copy = jsonl_files if jsonl_files else json_files

copied_files = []

for file_path in input_files_to_copy:
    target_path = os.path.join(STEP5_INPUT_DIR, file_path.name)
    shutil.copy(str(file_path), target_path)
    copied_files.append(file_path.name)

# ------------------------------------------------------------
# 6. Setup summary
# ------------------------------------------------------------

print("\nSynthetic Step5 setup complete.")
print("Synthetic experiment:", SYNTHETIC_EXPERIMENT_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("MAX_LENGTH:", MAX_LENGTH)
print("SOURCE_TYPE:", SOURCE_TYPE)
print("STEP5_INPUT_DIR:", STEP5_INPUT_DIR)
print("STEP5_OUTPUT_DIR:", STEP5_OUTPUT_DIR)
print("Synthetic Step4T zip:", step4_zip_path)
print("Number of synthetic Step4T files copied:", len(copied_files))

print("\nCopied files:")
for name in copied_files:
    print("-", name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Using synthetic Step4T zip from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/setting_synthetic/data_synthetic/pilot4_synthetic_step4_lr_5000.zip

Synthetic Step5 setup complete.
Synthetic experiment: pilot4_synthetic_step4_lr_5000
MODEL_NAME: Qwen/Qwen2.5-0.5B
MODEL_TAG: Qwen2_5_0_5B
MAX_LENGTH: 768
SOURCE_TYPE: pilot4_synthetic_step4_lr_5000_step4T_style_implicit_transitive_probe_samples
STEP5_INPUT_DIR: /content/pilot_4/data/pilot4_synthetic_step4_lr_5000_step4T_input
STEP5_OUTPUT_DIR: /content/pilot_4/data/pilot4_synthetic_step4_lr_5000_step5_hidden_states_qwen2_5_0_5b
Synthetic Step4T zip: /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/setting_synthetic/data_synthetic/pilot4_synthetic_step4_lr_5000.zip
Number of synthetic Step4T files copied: 1

Copied files:
- step4T_pilot4_

In [ ]:
# ============================================================
# Imports for Step 5
# ============================================================

import json
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Imports loaded.")
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("MAX_LENGTH:", MAX_LENGTH)
print("STEP5_INPUT_DIR:", STEP5_INPUT_DIR)
print("STEP5_OUTPUT_DIR:", STEP5_OUTPUT_DIR)

Imports loaded.
MODEL_NAME: Qwen/Qwen2.5-0.5B
MODEL_TAG: Qwen2_5_0_5B
MAX_LENGTH: 768
STEP5_INPUT_DIR: /content/pilot_4/data/pilot4_synthetic_step4_lr_5000_step4T_input
STEP5_OUTPUT_DIR: /content/pilot_4/data/pilot4_synthetic_step4_lr_5000_step5_hidden_states_qwen2_5_0_5b


In [ ]:
# ============================================================
# Load tokenizer and model
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        dtype = torch.bfloat16
        device_map = "auto"
    else:
        dtype = torch.float32
        device_map = None

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map=device_map,
    )

    model.eval()

    return tokenizer, model


tokenizer, model = load_model_and_tokenizer(MODEL_NAME)

print("Tokenizer and model loaded.")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model class:", model.__class__.__name__)
print("Number of hidden layers including embedding output:", model.config.num_hidden_layers + 1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Tokenizer and model loaded.
Tokenizer class: Qwen2Tokenizer
Model class: Qwen2ForCausalLM
Number of hidden layers including embedding output: 25


In [ ]:
# ============================================================
# Device / dtype check without reloading model
# ============================================================

print("CUDA environment check:")
print("torch.cuda.is_available():", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("CUDA allocated memory GB:", round(torch.cuda.memory_allocated(0) / 1024**3, 3))
    print("CUDA reserved memory GB:", round(torch.cuda.memory_reserved(0) / 1024**3, 3))
else:
    print("CUDA is not available. The runtime is using CPU.")

print("\nModel placement check:")

try:
    first_param = next(model.parameters())
    print("Model dtype:", first_param.dtype)
    print("Model device:", first_param.device)

    if first_param.device.type == "cuda":
        print("Model is on GPU.")
    else:
        print("Model is not on GPU.")

except NameError:
    print("model is not defined in this runtime. The model-loading cell has not been run.")

CUDA environment check:
torch.cuda.is_available(): True
CUDA device count: 1
CUDA device name: Tesla T4
CUDA capability: (7, 5)
CUDA allocated memory GB: 0.92
CUDA reserved memory GB: 0.932

Model placement check:
Model dtype: torch.bfloat16
Model device: cuda:0
Model is on GPU.


In [ ]:
# ============================================================
# Utility functions for Step4T implicit-transitive samples
# ============================================================

def load_step4_samples(path: str) -> List[Dict[str, Any]]:
    """
    Load Step4T probe samples from JSONL or JSON.
    """
    if path.endswith(".jsonl"):
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    if path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            return data

        raise ValueError(
            f"Expected a list in JSON file, got {type(data)}. path={path}"
        )

    raise ValueError(f"Unsupported file type: {path}")


def mean_span(
    token_hidden: torch.Tensor,
    start: int,
    end: int,
) -> torch.Tensor:
    """
    Mean-pool hidden states over token span [start, end).
    token_hidden shape: [seq_len, hidden_dim]
    """
    if start < 0 or end <= start or end > token_hidden.shape[0]:
        raise ValueError(
            f"Invalid token span: start={start}, end={end}, seq_len={token_hidden.shape[0]}"
        )

    return token_hidden[start:end].mean(dim=0)


def char_span_to_token_span(
    offsets: List[Tuple[int, int]],
    char_span: Dict[str, int],
) -> Optional[Tuple[int, int]]:
    """
    Convert character span to token span using tokenizer offset mapping.

    Returns:
        (token_start, token_end_exclusive)
    """
    if char_span is None:
        return None

    char_start = char_span.get("start")
    char_end = char_span.get("end")

    if char_start is None or char_end is None or char_end <= char_start:
        return None

    token_indices = []

    for tok_idx, offset in enumerate(offsets):
        tok_start, tok_end = offset

        # Skip special tokens or empty offsets.
        if tok_start == tok_end:
            continue

        overlap = min(tok_end, char_end) - max(tok_start, char_start)

        if overlap > 0:
            token_indices.append(tok_idx)

    if not token_indices:
        return None

    return (token_indices[0], token_indices[-1] + 1)


def validate_step4_sample(row: Dict[str, Any]) -> None:
    """
    Validate the minimum schema required by Step 5.

    This version is for Step4T implicit-transitive samples.
    It does not require explicit text_relation_label/text_subject_uid/text_object_uid,
    because the probe relation is implicit and supported by supporting_relations.
    """
    required_top_keys = {
        "sample_id",
        "pair_group_id",
        "scene",
        "paragraph_id",
        "text",
        "evidence",
        "probe_pair",
    }

    missing_top = [key for key in required_top_keys if key not in row]

    if missing_top:
        raise KeyError(
            f"Step4T sample missing required top-level keys: {missing_top}. "
            f"sample_id={row.get('sample_id')}"
        )

    probe_pair = row["probe_pair"]
    evidence = row["evidence"]

    required_probe_keys = {
        "probe_subject_uid",
        "probe_object_uid",
        "probe_relation_label",
        "probe_subject_span_in_paragraph",
        "probe_object_span_in_paragraph",
        "probe_subject_all_char_spans_in_paragraph",
        "probe_object_all_char_spans_in_paragraph",
    }

    missing_probe = [key for key in required_probe_keys if key not in probe_pair]

    if missing_probe:
        raise KeyError(
            f"Step4T sample missing required probe_pair keys: {missing_probe}. "
            f"sample_id={row.get('sample_id')}"
        )

    required_evidence_keys = {
        "evidence_type",
        "probe_direction_relative_to_text",
        "implicit_rule",
        "anchor_uid",
        "supporting_relations",
        "num_supporting_paths",
    }

    missing_evidence = [key for key in required_evidence_keys if key not in evidence]

    if missing_evidence:
        raise KeyError(
            f"Step4T sample missing required evidence keys: {missing_evidence}. "
            f"sample_id={row.get('sample_id')}"
        )

    if evidence.get("evidence_type") != "implicit_transitive":
        raise ValueError(
            f"Expected evidence_type='implicit_transitive', "
            f"got {evidence.get('evidence_type')}. "
            f"sample_id={row.get('sample_id')}"
        )


def pick_probe_span(
    probe_pair: Dict[str, Any],
    role: str,
) -> Optional[Dict[str, int]]:
    """
    Select character span for probe subject/object.

    Priority:
        1. probe-specific span in paragraph
        2. first all-span occurrence in paragraph
    """
    if role == "subject":
        span = probe_pair.get("probe_subject_span_in_paragraph")
        all_spans = probe_pair.get("probe_subject_all_char_spans_in_paragraph", [])
    elif role == "object":
        span = probe_pair.get("probe_object_span_in_paragraph")
        all_spans = probe_pair.get("probe_object_all_char_spans_in_paragraph", [])
    else:
        raise ValueError(f"Unknown role: {role}")

    if span is not None:
        return span

    if all_spans:
        return all_spans[0]

    return None


def discover_step4_input_files() -> List[str]:
    """
    Prefer JSONL files. If no JSONL exists, use JSON files.
    This version searches Step4T implicit-transitive file names.
    """
    jsonl_paths = sorted(
        Path(STEP5_INPUT_DIR).glob("step4T_*_implicit_transitive_probe_samples.jsonl")
    )

    if jsonl_paths:
        return [str(p) for p in jsonl_paths]

    json_paths = sorted(
        Path(STEP5_INPUT_DIR).glob("step4T_*_implicit_transitive_probe_samples.json")
    )

    if json_paths:
        return [str(p) for p in json_paths]

    raise FileNotFoundError(
        f"No Step4T implicit-transitive probe sample files found in {STEP5_INPUT_DIR}"
    )


print("Utility functions loaded for Step4T implicit-transitive samples.")

Utility functions loaded for Step4T implicit-transitive samples.


In [ ]:
# ============================================================
# Build Step 5 examples from Step4T implicit-transitive probe samples
#
# Important:
#   Step 5 must use probe_pair subject/object, not supporting_relation subject/object.
#
# For Step4T:
#   The target relation is implicit.
#   The supporting_relations explain why this relation can be inferred.
#
# Therefore the feature must be:
#   hidden(probe_subject) - hidden(probe_object)
# ============================================================

def build_step5_example_from_step4_sample(row: Dict[str, Any]) -> Dict[str, Any]:
    validate_step4_sample(row)

    text = row["text"]
    probe_pair = row["probe_pair"]
    evidence = row["evidence"]

    probe_subject_uid = probe_pair["probe_subject_uid"]
    probe_object_uid = probe_pair["probe_object_uid"]
    probe_relation_label = probe_pair["probe_relation_label"]

    evidence_type = evidence.get("evidence_type")

    subject_char_span = pick_probe_span(probe_pair, role="subject")
    object_char_span = pick_probe_span(probe_pair, role="object")

    return {
        "sample_id": row["sample_id"],
        "pair_group_id": row["pair_group_id"],

        "scene": row.get("scene"),
        "paragraph_id": row.get("paragraph_id"),
        "chunk_id": row.get("chunk_id"),
        "cluster_id": row.get("cluster_id"),
        "grid_i": row.get("grid_i"),
        "grid_j": row.get("grid_j"),
        "paragraph_index_within_cluster": row.get("paragraph_index_within_cluster"),
        "generation_mode": row.get("generation_mode"),
        "experiment": row.get("experiment"),

        # Full LLM input.
        "text": text,

        # Label for later linear probe.
        "relation": probe_relation_label,

        # Probe subject/object direction.
        "subject_alias": probe_subject_uid,
        "object_alias": probe_object_uid,
        "subject_id": probe_pair.get("probe_subject_id"),
        "object_id": probe_pair.get("probe_object_id"),
        "subject_type": probe_pair.get("probe_subject_type"),
        "object_type": probe_pair.get("probe_object_type"),
        "subject_mention_text": probe_pair.get("probe_subject_mention_text"),
        "object_mention_text": probe_pair.get("probe_object_mention_text"),

        # Character spans for hidden-state extraction.
        "subject_char_span": subject_char_span,
        "object_char_span": object_char_span,
        "subject_all_char_spans": probe_pair.get("probe_subject_all_char_spans_in_paragraph"),
        "object_all_char_spans": probe_pair.get("probe_object_all_char_spans_in_paragraph"),

        # Probe-pair metadata.
        "label_space": probe_pair.get("label_space"),
        "has_relation_label": probe_pair.get("has_relation_label"),
        "is_explicit_in_text": probe_pair.get("is_explicit_in_text"),
        "is_inverse_of_text_relation": probe_pair.get("is_inverse_of_text_relation"),
        "is_symmetric_relation": probe_pair.get("is_symmetric_relation"),
        "is_directional_relation": probe_pair.get("is_directional_relation"),
        "is_implicit_transitive": probe_pair.get("is_implicit_transitive"),

        # Evidence metadata.
        "evidence_type": evidence_type,
        "probe_direction_relative_to_text": evidence.get("probe_direction_relative_to_text"),
        "has_explicit_evidence_sentence": evidence.get("has_explicit_evidence_sentence"),
        "evidence_sentence": evidence.get("evidence_sentence"),
        "sentence_index_in_paragraph": evidence.get("sentence_index_in_paragraph"),
        "surface_order": evidence.get("surface_order"),

        # Step4T implicit-transitive metadata.
        "supporting_relation_source": evidence.get("supporting_relation_source"),
        "implicit_rule": evidence.get("implicit_rule"),
        "anchor_uid": evidence.get("anchor_uid"),
        "supporting_relations": evidence.get("supporting_relations"),
        "num_supporting_paths": evidence.get("num_supporting_paths"),
        "original_pair_evidence_type": evidence.get("original_pair_evidence_type"),
        "label_source": evidence.get("label_source"),
        "all_supporting_paths": evidence.get("all_supporting_paths"),

        # Additional source metadata.
        "source_relation": row.get("source_relation"),
        "geometry": row.get("geometry"),
        "source_step3_file": row.get("source_step3_file"),
        "source_step3_scene": row.get("source_step3_scene"),
    }


def build_step5_examples(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    examples = []

    for row in rows:
        ex = build_step5_example_from_step4_sample(row)
        examples.append(ex)

    return examples


print("Step 5 example builder loaded for Step4T implicit-transitive samples.")

Step 5 example builder loaded for Step4T implicit-transitive samples.


In [ ]:
# ============================================================
# Preflight check: token length and probe span coverage
#
# This checks whether MAX_LENGTH is sufficient before hidden-state extraction.
# It verifies that probe_subject/probe_object spans remain visible after truncation.
# ============================================================

def check_token_length_and_span_coverage(
    input_files: List[str],
    tokenizer,
    max_length: int = MAX_LENGTH,
    max_bad_examples: int = 10,
) -> Dict[str, Any]:
    total_examples = 0
    token_lengths_no_trunc = []
    truncated_count = 0
    span_failed_count = 0
    bad_examples = []

    for input_path in input_files:
        rows = load_step4_samples(input_path)
        examples = build_step5_examples(rows)

        for ex in examples:
            total_examples += 1
            text = ex["text"]

            # Tokenize once without truncation to get true length.
            enc_full = tokenizer(
                text,
                return_tensors="pt",
                truncation=False,
                return_offsets_mapping=True,
                add_special_tokens=True,
            )

            true_len = int(enc_full["input_ids"].shape[1])
            token_lengths_no_trunc.append(true_len)

            if true_len > max_length:
                truncated_count += 1

            # Tokenize with actual MAX_LENGTH setting used in extraction.
            enc_trunc = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                return_offsets_mapping=True,
                add_special_tokens=True,
            )

            offsets = enc_trunc["offset_mapping"][0].tolist()

            subj_span = ex.get("subject_char_span")
            obj_span = ex.get("object_char_span")

            subj_tok_span = char_span_to_token_span(offsets, subj_span)
            obj_tok_span = char_span_to_token_span(offsets, obj_span)

            if subj_tok_span is None or obj_tok_span is None:
                span_failed_count += 1

                if len(bad_examples) < max_bad_examples:
                    bad_examples.append({
                        "sample_id": ex.get("sample_id"),
                        "source_file": os.path.basename(input_path),
                        "scene": ex.get("scene"),
                        "paragraph_id": ex.get("paragraph_id"),
                        "relation": ex.get("relation"),
                        "evidence_type": ex.get("evidence_type"),
                        "probe_direction_relative_to_text": ex.get(
                            "probe_direction_relative_to_text"
                        ),
                        "true_token_length": true_len,
                        "max_length": max_length,
                        "subject_alias": ex.get("subject_alias"),
                        "object_alias": ex.get("object_alias"),
                        "subject_char_span": subj_span,
                        "object_char_span": obj_span,
                        "subject_token_span_after_truncation": subj_tok_span,
                        "object_token_span_after_truncation": obj_tok_span,
                    })

    if token_lengths_no_trunc:
        sorted_lens = sorted(token_lengths_no_trunc)
        n = len(sorted_lens)

        def percentile(p: float) -> int:
            idx = min(n - 1, int(round((p / 100) * (n - 1))))
            return sorted_lens[idx]

        summary = {
            "total_examples": total_examples,
            "max_length": max_length,
            "token_length_min": min(sorted_lens),
            "token_length_max": max(sorted_lens),
            "token_length_p50": percentile(50),
            "token_length_p95": percentile(95),
            "token_length_p99": percentile(99),
            "num_examples_longer_than_max_length": truncated_count,
            "num_examples_with_failed_probe_span_after_truncation": span_failed_count,
            "bad_examples": bad_examples,
        }
    else:
        summary = {
            "total_examples": 0,
            "max_length": max_length,
            "token_length_min": None,
            "token_length_max": None,
            "token_length_p50": None,
            "token_length_p95": None,
            "token_length_p99": None,
            "num_examples_longer_than_max_length": 0,
            "num_examples_with_failed_probe_span_after_truncation": 0,
            "bad_examples": [],
        }

    return summary


step4_input_files = discover_step4_input_files()

preflight_summary = check_token_length_and_span_coverage(
    input_files=step4_input_files,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

print("Preflight token/span coverage summary:")
print(json.dumps(preflight_summary, indent=2, ensure_ascii=False))

if preflight_summary["num_examples_with_failed_probe_span_after_truncation"] > 0:
    raise ValueError(
        "Some probe subject/object spans are not visible after tokenization/truncation. "
        "Increase MAX_LENGTH, for example to 768 or 1024."
    )

Preflight token/span coverage summary:
{
  "total_examples": 5000,
  "max_length": 768,
  "token_length_min": 88,
  "token_length_max": 172,
  "token_length_p50": 125,
  "token_length_p95": 156,
  "token_length_p99": 162,
  "num_examples_longer_than_max_length": 0,
  "num_examples_with_failed_probe_span_after_truncation": 0,
  "bad_examples": []
}


In [ ]:
# ============================================================
# Hidden-state extraction for one Step 5 example
# ============================================================

@torch.no_grad()
def process_probe_example(
    ex: Dict[str, Any],
    tokenizer,
    model,
    max_length: int = MAX_LENGTH,
) -> Tuple[Optional[Dict[str, Any]], Optional[Dict[str, Any]]]:
    """
    Process one Step 5 probe example.

    Returns:
        (record, skip_info)

    record is None if the example is skipped.
    """
    text = ex["text"]
    subject_alias = ex["subject_alias"]
    object_alias = ex["object_alias"]

    subject_char_span = ex.get("subject_char_span")
    object_char_span = ex.get("object_char_span")

    if subject_char_span is None or object_char_span is None:
        return None, {
            "sample_id": ex.get("sample_id"),
            "reason": "missing_subject_or_object_char_span",
            "subject_alias": subject_alias,
            "object_alias": object_alias,
            "subject_char_span": subject_char_span,
            "object_char_span": object_char_span,
        }

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )

    offsets = enc["offset_mapping"][0].tolist()

    subject_token_span = char_span_to_token_span(offsets, subject_char_span)
    object_token_span = char_span_to_token_span(offsets, object_char_span)

    if subject_token_span is None or object_token_span is None:
        return None, {
            "sample_id": ex.get("sample_id"),
            "reason": "char_span_to_token_span_failed_or_truncated",
            "text_length_chars": len(text),
            "max_length": max_length,
            "subject_alias": subject_alias,
            "object_alias": object_alias,
            "subject_char_span": subject_char_span,
            "object_char_span": object_char_span,
            "subject_token_span": subject_token_span,
            "object_token_span": object_token_span,
        }

    # Remove offset_mapping before passing to model.
    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    model_device = next(model.parameters()).device

    input_ids = input_ids.to(model_device)
    attention_mask = attention_mask.to(model_device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )

    hidden_states = outputs.hidden_states

    layer_subject_vectors = []
    layer_object_vectors = []
    layer_diff_features = []
    layer_concat_features = []

    for layer_h in hidden_states:
        # layer_h shape: [1, seq_len, hidden_dim]
        token_h = layer_h.squeeze(0).detach().cpu()

        h_subject = mean_span(
            token_h,
            subject_token_span[0],
            subject_token_span[1],
        )

        h_object = mean_span(
            token_h,
            object_token_span[0],
            object_token_span[1],
        )

        diff = h_subject - h_object
        concat = torch.cat([h_subject, h_object], dim=0)

        layer_subject_vectors.append(h_subject)
        layer_object_vectors.append(h_object)
        layer_diff_features.append(diff)
        layer_concat_features.append(concat)

    decoded_tokens = tokenizer.convert_ids_to_tokens(
        enc["input_ids"][0].tolist()
    )

    record = {
        "sample_id": ex["sample_id"],
        "pair_group_id": ex["pair_group_id"],

        "scene": ex.get("scene"),
        "paragraph_id": ex.get("paragraph_id"),
        "chunk_id": ex.get("chunk_id"),
        "cluster_id": ex.get("cluster_id"),
        "grid_i": ex.get("grid_i"),
        "grid_j": ex.get("grid_j"),
        "paragraph_index_within_cluster": ex.get("paragraph_index_within_cluster"),
        "generation_mode": ex.get("generation_mode"),
        "experiment": ex.get("experiment"),

        "text": text,
        "relation": ex["relation"],

        # Probe subject/object direction.
        "subject_alias": subject_alias,
        "object_alias": object_alias,
        "subject_id": ex.get("subject_id"),
        "object_id": ex.get("object_id"),
        "subject_type": ex.get("subject_type"),
        "object_type": ex.get("object_type"),
        "subject_mention_text": ex.get("subject_mention_text"),
        "object_mention_text": ex.get("object_mention_text"),

        "subject_char_span": subject_char_span,
        "object_char_span": object_char_span,
        "subject_token_span": subject_token_span,
        "object_token_span": object_token_span,

        "subject_all_char_spans": ex.get("subject_all_char_spans"),
        "object_all_char_spans": ex.get("object_all_char_spans"),

                # Probe-pair metadata.
        "label_space": ex.get("label_space"),
        "has_relation_label": ex.get("has_relation_label"),
        "is_explicit_in_text": ex.get("is_explicit_in_text"),
        "is_inverse_of_text_relation": ex.get("is_inverse_of_text_relation"),
        "is_symmetric_relation": ex.get("is_symmetric_relation"),
        "is_directional_relation": ex.get("is_directional_relation"),
        "is_implicit_transitive": ex.get("is_implicit_transitive"),

        # Evidence metadata for later grouping.
        "evidence_type": ex.get("evidence_type"),
        "probe_direction_relative_to_text": ex.get("probe_direction_relative_to_text"),
        "has_explicit_evidence_sentence": ex.get("has_explicit_evidence_sentence"),
        "evidence_sentence": ex.get("evidence_sentence"),
        "sentence_index_in_paragraph": ex.get("sentence_index_in_paragraph"),
        "surface_order": ex.get("surface_order"),

        # Step4T implicit-transitive metadata.
        "supporting_relation_source": ex.get("supporting_relation_source"),
        "implicit_rule": ex.get("implicit_rule"),
        "anchor_uid": ex.get("anchor_uid"),
        "supporting_relations": ex.get("supporting_relations"),
        "num_supporting_paths": ex.get("num_supporting_paths"),
        "original_pair_evidence_type": ex.get("original_pair_evidence_type"),
        "label_source": ex.get("label_source"),
        "all_supporting_paths": ex.get("all_supporting_paths"),

        "source_relation": ex.get("source_relation"),
        "geometry": ex.get("geometry"),
        "source_step3_file": ex.get("source_step3_file"),
        "source_step3_scene": ex.get("source_step3_scene"),

        # Tokenization metadata.
        "input_ids": enc["input_ids"][0].cpu(),
        "attention_mask": enc["attention_mask"][0].cpu(),
        "offset_mapping": offsets,
        "decoded_tokens": decoded_tokens,
        "num_tokens": len(decoded_tokens),
        "max_length": max_length,

        # Hidden-state features.
        "layer_subject_vectors": torch.stack(layer_subject_vectors, dim=0),
        "layer_object_vectors": torch.stack(layer_object_vectors, dim=0),
        "layer_diff_features": torch.stack(layer_diff_features, dim=0),
        "layer_concat_features": torch.stack(layer_concat_features, dim=0),
    }

    return record, None


print("Hidden-state extraction function loaded.")

Hidden-state extraction function loaded.


In [ ]:
# ============================================================
# Process one Step4T file
# ============================================================

def process_step4_file(
    input_path: str,
    tokenizer,
    model,
    output_dir: str,
) -> Dict[str, Any]:
    rows = load_step4_samples(input_path)
    examples = build_step5_examples(rows)

    source_file = os.path.basename(input_path)

    # Infer scene from records if available.
    scenes = sorted({ex.get("scene") for ex in examples if ex.get("scene") is not None})
    scene = scenes[0] if len(scenes) == 1 else "mixed_scenes"

    kept_records = []
    skipped_records = []

    for ex in tqdm(examples, desc=f"Extracting hidden states: {source_file}"):
        record, skip_info = process_probe_example(
            ex,
            tokenizer=tokenizer,
            model=model,
            max_length=MAX_LENGTH,
        )

        if record is not None:
            kept_records.append(record)
        else:
            skipped_records.append(skip_info)

    if kept_records:
        num_layers = kept_records[0]["layer_diff_features"].shape[0]
        feature_dim = kept_records[0]["layer_diff_features"].shape[1]
        concat_dim = kept_records[0]["layer_concat_features"].shape[1]
    else:
        num_layers = None
        feature_dim = None
        concat_dim = None

    output_name = source_file.replace("_implicit_transitive_probe_samples.jsonl", "")
    output_name = output_name.replace("_implicit_transitive_probe_samples.json", "")

    output_pt_path = os.path.join(
        output_dir,
        f"{output_name}_step5_hidden_states_{MODEL_TAG}.pt"
    )

    summary = {
        "source_file": source_file,
        "scene": scene,
        "source_type": SOURCE_TYPE,
        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "input_text_version": "pilot4_synthetic_5000_implicit_lr_probe_samples",
        "input_text_mode": INPUT_TEXT_MODE,
        "feature_anchor": FEATURE_ANCHOR,

        "run_mode": RUN_MODE,
        "max_length": MAX_LENGTH,

        "num_input_examples": len(examples),
        "num_kept_examples": len(kept_records),
        "num_skipped_examples": len(skipped_records),

        "num_layers": num_layers,
        "feature_dim": feature_dim,
        "concat_dim": concat_dim,

        "label_counts": dict(Counter(r["relation"] for r in kept_records)),
        "evidence_type_counts": dict(Counter(r["evidence_type"] for r in kept_records)),
        "implicit_rule_counts": dict(Counter(r.get("implicit_rule") for r in kept_records)),
        "probe_direction_counts": dict(
            Counter(r["probe_direction_relative_to_text"] for r in kept_records)
        ),
        "skip_reason_counts": dict(Counter(s["reason"] for s in skipped_records)),
    }

    payload = {
        **summary,
        "records": kept_records,
        "skipped_records": skipped_records,
    }

    torch.save(payload, output_pt_path)

    summary["output_pt_path"] = output_pt_path

    return summary


print("File processing function loaded for Step4T implicit-transitive samples.")

File processing function loaded for Step4T implicit-transitive samples.


In [ ]:
# ============================================================
# Run Step 5 hidden-state extraction
# ============================================================

step4_input_files = discover_step4_input_files()

print("Discovered Step 4 input files:", len(step4_input_files))
for path in step4_input_files:
    print("-", os.path.basename(path))

all_file_summaries = []

for input_path in step4_input_files:
    summary = process_step4_file(
        input_path=input_path,
        tokenizer=tokenizer,
        model=model,
        output_dir=STEP5_OUTPUT_DIR,
    )
    all_file_summaries.append(summary)

print("\nStep 5 processing complete.")
print("Processed files:", len(all_file_summaries))

if PRINT_FILE_SUMMARY:
    for summary in all_file_summaries:
        print(
            f"- {summary['source_file']} | "
            f"scene={summary['scene']} | "
            f"kept={summary['num_kept_examples']} | "
            f"skipped={summary['num_skipped_examples']} | "
            f"output={os.path.basename(summary['output_pt_path'])}"
        )

Discovered Step 4 input files: 1
- step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json


Extracting hidden states: step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json:   0%| …


Step 5 processing complete.
Processed files: 1


NameError: name 'PRINT_FILE_SUMMARY' is not defined

In [ ]:
# ============================================================
# Print Step 5 processing summary without rerunning extraction
# ============================================================

PRINT_FILE_SUMMARY = True

print("\nStep 5 processing complete.")
print("Processed files:", len(all_file_summaries))

if PRINT_FILE_SUMMARY:
    for summary in all_file_summaries:
        print(
            f"- {summary['source_file']} | "
            f"scene={summary['scene']} | "
            f"kept={summary['num_kept_examples']} | "
            f"skipped={summary['num_skipped_examples']} | "
            f"output={os.path.basename(summary['output_pt_path'])}"
        )


Step 5 processing complete.
Processed files: 1
- step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json | scene=mixed_scenes | kept=5000 | skipped=0 | output=step4T_pilot4_synthetic_step4_lr_5000_step5_hidden_states_Qwen2_5_0_5B.pt


In [ ]:
# ============================================================
# Save Step 5 export summary
# ============================================================

summary_path = os.path.join(
    STEP5_OUTPUT_DIR,
    f"synthetic_5000_step5_summary_{MODEL_TAG}.json"
)

source_step4_files = [
    os.path.basename(path)
    for path in step4_input_files
]

summary_payload = {
    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "source_type": SOURCE_TYPE,
    "input_text_mode": INPUT_TEXT_MODE,
    "feature_anchor": FEATURE_ANCHOR,
    "run_mode": RUN_MODE,
    "max_length": MAX_LENGTH,
    "step5_input_dir": STEP5_INPUT_DIR,
    "step5_output_dir": STEP5_OUTPUT_DIR,

    # Actual Step 4 files discovered and processed by this Step 5 run.
    "source_step4_files": source_step4_files,

    "num_processed_files": len(all_file_summaries),
    "file_summaries": all_file_summaries,
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, ensure_ascii=False, indent=2)

print("Step 5 summary written to:")
print(summary_path)

print("\nSource Step 4 files processed:")
for name in source_step4_files:
    print("-", name)

print("\nOverall kept examples:", sum(s["num_kept_examples"] for s in all_file_summaries))
print("Overall skipped examples:", sum(s["num_skipped_examples"] for s in all_file_summaries))

Step 5 summary written to:
/content/pilot_4/data/pilot4_synthetic_step4_lr_5000_step5_hidden_states_qwen2_5_0_5b/synthetic_5000_step5_summary_Qwen2_5_0_5B.json

Source Step 4 files processed:
- step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json

Overall kept examples: 5000
Overall skipped examples: 0


In [ ]:
# ============================================================
# Sanity checks for Step 5 outputs
# ============================================================

pt_files = sorted(Path(STEP5_OUTPUT_DIR).glob("*_step5_hidden_states_*.pt"))

if not pt_files:
    raise FileNotFoundError(f"No Step 5 .pt files found in {STEP5_OUTPUT_DIR}")

print("Step 5 .pt files:", len(pt_files))
for pt_path in pt_files:
    print("-", pt_path.name)

# Load first file for inspection.
sample_payload = torch.load(pt_files[0], map_location="cpu")

print("\nLoaded sample payload:")
print("source_file:", sample_payload.get("source_file"))
print("scene:", sample_payload.get("scene"))
print("num_kept_examples:", sample_payload.get("num_kept_examples"))
print("num_skipped_examples:", sample_payload.get("num_skipped_examples"))
print("num_layers:", sample_payload.get("num_layers"))
print("feature_dim:", sample_payload.get("feature_dim"))
print("concat_dim:", sample_payload.get("concat_dim"))
print("label_counts:", sample_payload.get("label_counts"))
print("evidence_type_counts:", sample_payload.get("evidence_type_counts"))
print("implicit_rule_counts:", sample_payload.get("implicit_rule_counts"))

records = sample_payload.get("records", [])

if records:
    r0 = records[0]

    print("\nFirst record keys:")
    print(sorted(r0.keys()))

    print("\nFirst record metadata:")
    print("sample_id:", r0["sample_id"])
    print("pair_group_id:", r0["pair_group_id"])
    print("relation:", r0["relation"])
    print("subject_alias:", r0["subject_alias"])
    print("object_alias:", r0["object_alias"])
    print("subject_type:", r0["subject_type"])
    print("object_type:", r0["object_type"])
    print("evidence_type:", r0["evidence_type"])
    print("probe_direction_relative_to_text:", r0["probe_direction_relative_to_text"])
    print("is_implicit_transitive:", r0["is_implicit_transitive"])
    print("implicit_rule:", r0["implicit_rule"])
    print("anchor_uid:", r0["anchor_uid"])
    print("num_supporting_paths:", r0["num_supporting_paths"])

    print("\nSupporting relations:")
    for sr in r0.get("supporting_relations", []):
        print(
            "-",
            sr.get("subject_uid"),
            sr.get("relation"),
            sr.get("object_uid"),
            "| sentence:",
            sr.get("sentence"),
        )

    print("\nFeature shapes:")
    print("layer_subject_vectors:", tuple(r0["layer_subject_vectors"].shape))
    print("layer_object_vectors:", tuple(r0["layer_object_vectors"].shape))
    print("layer_diff_features:", tuple(r0["layer_diff_features"].shape))
    print("layer_concat_features:", tuple(r0["layer_concat_features"].shape))

    print("\nSpan check:")
    print("subject_char_span:", r0["subject_char_span"])
    print("object_char_span:", r0["object_char_span"])
    print("subject_token_span:", r0["subject_token_span"])
    print("object_token_span:", r0["object_token_span"])

    print("\nFeature direction check:")
    print(
        "Feature direction is always: "
        f"hidden({r0['subject_alias']}) - hidden({r0['object_alias']})"
    )
else:
    print("No kept records found in first payload.")

Step 5 .pt files: 1
- step4T_pilot4_synthetic_step4_lr_5000_step5_hidden_states_Qwen2_5_0_5B.pt

Loaded sample payload:
source_file: step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json
scene: mixed_scenes
num_kept_examples: 5000
num_skipped_examples: 0
num_layers: 25
feature_dim: 896
concat_dim: 1792
label_counts: {'right_of': 2501, 'left_of': 2499}
evidence_type_counts: {'implicit_transitive': 5000}
implicit_rule_counts: {'shared_anchor_opposite_sides': 1667, 'chain_same_direction': 1666, 'anchor_between_reversed_surface_form': 1667}

First record keys:
['all_supporting_paths', 'anchor_uid', 'attention_mask', 'chunk_id', 'cluster_id', 'decoded_tokens', 'evidence_sentence', 'evidence_type', 'experiment', 'generation_mode', 'geometry', 'grid_i', 'grid_j', 'has_explicit_evidence_sentence', 'has_relation_label', 'implicit_rule', 'input_ids', 'is_directional_relation', 'is_explicit_in_text', 'is_implicit_transitive', 'is_inverse_of_text_relation', 'is_symmetric_rel

In [ ]:
# ============================================================
# Sanity check for implicit-transitive samples
#
# Step4T target relation is not explicitly stated as one sentence.
# It is inferred from supporting_relations.
#
# Step 5 must still extract:
#   hidden(probe_subject) - hidden(probe_object)
# ============================================================

implicit_examples = []

for pt_path in pt_files:
    payload = torch.load(pt_path, map_location="cpu")
    for row in payload.get("records", []):
        if row.get("evidence_type") == "implicit_transitive":
            implicit_examples.append(row)

print("Number of implicit-transitive examples found:", len(implicit_examples))

if implicit_examples:
    ex = implicit_examples[0]

    print("\nFirst implicit-transitive example:")
    print("sample_id:", ex["sample_id"])
    print("pair_group_id:", ex["pair_group_id"])
    print("relation / probe label:", ex["relation"])
    print("probe subject:", ex["subject_alias"])
    print("probe object:", ex["object_alias"])
    print("implicit_rule:", ex["implicit_rule"])
    print("anchor_uid:", ex["anchor_uid"])
    print("num_supporting_paths:", ex["num_supporting_paths"])

    print("\nSupporting relations:")
    for sr in ex.get("supporting_relations", []):
        print(
            "-",
            sr.get("subject_uid"),
            sr.get("relation"),
            sr.get("object_uid"),
            "| sentence:",
            sr.get("sentence"),
        )

    print("\nSpan check:")
    print("subject_char_span:", ex["subject_char_span"])
    print("object_char_span:", ex["object_char_span"])
    print("subject_token_span:", ex["subject_token_span"])
    print("object_token_span:", ex["object_token_span"])

    print("\nInterpretation check:")
    print(
        "Feature direction is always: "
        f"hidden({ex['subject_alias']}) - hidden({ex['object_alias']})"
    )
else:
    print("No implicit-transitive examples found. Check Step4T input files.")

Number of implicit-transitive examples found: 5000

First implicit-transitive example:
sample_id: synthetic_step4T_000000_1ee05ed6
pair_group_id: synthetic_FloorPlan2_paragraph_000000|house_plant_000602|pan_000924|right_of|implicit_transitive|shared_anchor_opposite_sides
relation / probe label: right_of
probe subject: house_plant_000602
probe object: pan_000924
implicit_rule: shared_anchor_opposite_sides
anchor_uid: stove_burner_000668
num_supporting_paths: 1

Supporting relations:
- house_plant_000602 right_of stove_burner_000668 | sentence: house_plant_000602 is on the right side of stove_burner_000668.
- pan_000924 left_of stove_burner_000668 | sentence: pan_000924 is on the left side of stove_burner_000668.

Span check:
subject_char_span: {'start': 222, 'end': 240}
object_char_span: {'start': 78, 'end': 88}
subject_token_span: (99, 109)
object_token_span: (29, 37)

Interpretation check:
Feature direction is always: hidden(house_plant_000602) - hidden(pan_000924)


In [ ]:
# ============================================================
# Inspect skipped examples, if any
# ============================================================

all_skipped = []

for pt_path in pt_files:
    payload = torch.load(pt_path, map_location="cpu")
    for skip in payload.get("skipped_records", []):
        skip = dict(skip)
        skip["pt_file"] = pt_path.name
        all_skipped.append(skip)

print("Total skipped examples:", len(all_skipped))

if all_skipped:
    print("\nSkip reason counts:")
    print(json.dumps(dict(Counter(s["reason"] for s in all_skipped)), indent=2, ensure_ascii=False))

    if PRINT_SKIP_EXAMPLES:
        print(f"\nFirst {min(MAX_SKIP_EXAMPLES, len(all_skipped))} skipped examples:")
        for skip in all_skipped[:MAX_SKIP_EXAMPLES]:
            print(json.dumps(skip, indent=2, ensure_ascii=False))

Total skipped examples: 0


In [ ]:
# ============================================================
# Zip Step 5 outputs for download
# ============================================================

from google.colab import files

output_files = sorted(os.listdir(STEP5_OUTPUT_DIR))

if not output_files:
    raise FileNotFoundError(
        f"No files found in STEP5_OUTPUT_DIR: {STEP5_OUTPUT_DIR}"
    )

zip_base = f"/content/pilot4_synthetic_lr_5000_step5_{MODEL_TAG}"

zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP5_OUTPUT_DIR,
)

print("Created zip:", zip_path)
print("STEP5_OUTPUT_DIR:", STEP5_OUTPUT_DIR)
print("Number of files included:", len(output_files))

print("\nFiles included:")
for name in output_files:
    print("-", name)

files.download(zip_path)

Created zip: /content/pilot4_synthetic_lr_5000_step5_Qwen2_5_0_5B.zip
STEP5_OUTPUT_DIR: /content/pilot_4/data/pilot4_synthetic_step4_lr_5000_step5_hidden_states_qwen2_5_0_5b
Number of files included: 2

Files included:
- step4T_pilot4_synthetic_step4_lr_5000_step5_hidden_states_Qwen2_5_0_5B.pt
- synthetic_5000_step5_summary_Qwen2_5_0_5B.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>